# OpenFOAM CFD Setup — Toroidal Propeller Simulation

This notebook:
1. Installs OpenFOAM v2212 on Google Colab (Ubuntu Linux only — see note below)
2. Generates propeller STL geometry
3. Creates OpenFOAM case directories
4. Runs a representative case (medium-gap toroidal, 4000 RPM)
5. Post-processes forces and compares with BEMT / experimental data

> **⚠️ WINDOWS USERS:** OpenFOAM does NOT run on Windows natively.
> Running cell 2 will detect this and print step-by-step instructions for
> using **Google Colab** (recommended) or **WSL2**.
> The BEMT notebook (`01_BEMT_Analysis.ipynb`) **does** work on Windows.

---

## Experimental setup being reproduced

The experiments in the Palileo & Sabanal (2024) thesis were conducted in a
**Westenberg Engineering WT 8600100-E Eiffel-type open-jet subsonic wind tunnel**
located at PAG-ASA Davao:

| Parameter | Value |
|---|---|
| Tunnel type | Eiffel-type (open-circuit, open-jet) |
| Jet outlet diameter | **600 mm** |
| Max flow speed | 80 m/s |
| Tested speeds | 3, 9, 15 m/s |
| Propeller diameter | 203.2 mm (8 in) |
| Blockage ratio | ~9% (propeller disk / jet cross-section) |

The `wind_tunnel_openjet` case type in this notebook replicates this geometry:
the domain is 600 × 600 mm in cross-section (matching the jet), with slip lateral
boundaries approximating the free-jet shear layer.  This is physically more
accurate than a free-field simulation because the 9% blockage is significant.

---
**Solver:** simpleFoam (steady-state RANS)  
**Turbulence:** k-ω SST  
**Rotation:** Multiple Reference Frame (MRF)

In [ ]:
import os, sys

REPO_URL = 'https://github.com/rp3gregorio/Propeller-CFD.git'
BRANCH   = 'claude/add-cfd-propeller-simulation-EEQqr'

# ── Detect environment ──────────────────────────────────────────────────
ON_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if ON_COLAB:
    REPO_DIR = '/content/Propeller-CFD'
    if not os.path.isdir(REPO_DIR):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {BRANCH}
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    # Running locally — find the repo root by looking for bemt/ directory
    _here = os.path.abspath(os.getcwd())
    for _candidate in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_candidate, 'bemt')):
            REPO_DIR = _candidate
            break
    else:
        raise RuntimeError(
            "Could not find the repo root.\n"
            "Open Jupyter from inside the cloned Propeller-CFD folder."
        )
    print(f'Running locally. Repo root: {REPO_DIR}')

# Clear any cached Python imports
for _mod in [m for m in sys.modules if m.startswith(('bemt', 'geometry', 'postprocessing'))]:
    del sys.modules[_mod]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess as _sp
print('Working directory:', os.getcwd())
print('Branch:', _sp.run('git branch --show-current', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())
print('Latest commit:', _sp.run('git log --oneline -1', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())

In [ ]:
import subprocess, os, sys, platform

# ── WINDOWS CHECK ─────────────────────────────────────────────────────────────
if platform.system() == 'Windows':
    raise RuntimeError(
        "\n"
        "╔══════════════════════════════════════════════════════════════════════╗\n"
        "║  OpenFOAM does NOT run on Windows natively.                         ║\n"
        "║                                                                      ║\n"
        "║  OPTION A — Google Colab (easiest, free):                            ║\n"
        "║    1. Go to https://colab.research.google.com                        ║\n"
        "║    2. File → Open notebook → GitHub tab                              ║\n"
        "║    3. Paste: rp3gregorio/Propeller-CFD                               ║\n"
        "║    4. Select branch: claude/add-cfd-propeller-simulation-EEQqr       ║\n"
        "║    5. Open: notebooks/02_OpenFOAM_Setup.ipynb                        ║\n"
        "║                                                                      ║\n"
        "║  The BEMT notebook (01_BEMT_Analysis.ipynb) works on Windows.       ║\n"
        "╚══════════════════════════════════════════════════════════════════════╝\n"
    )

# ── Already installed check ────────────────────────────────────────────────────
def _run(cmd, check=True, cwd=None):
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f'Command failed (rc={r.returncode}):\n{r.stdout[-2000:]}')
    return r

if _run('command -v blockMesh', check=False).returncode == 0 or \
   os.path.isfile('/usr/lib/openfoam/openfoam2212/bin/blockMesh'):
    print('OpenFOAM already installed — skipping.')
else:
    print('--- Installing OpenFOAM v2212 via official repo setup script ---')

    # Step 1: Download the official repo setup script to a temp file
    # (avoids the piped-curl approach which can fail if the server returns HTML)
    _run('curl -fsSL https://dl.openfoam.com/add-debian-repo.sh -o /tmp/add-of-repo.sh')

    # Step 2: Verify it looks like a shell script, not an error page
    with open('/tmp/add-of-repo.sh') as f:
        first_line = f.readline().strip()
    if not first_line.startswith('#'):
        raise RuntimeError(
            f'Downloaded script does not look like a shell script '
            f'(first line: {first_line!r}).\n'
            'Check your network connection / Colab internet access.'
        )

    # Step 3: Run the setup script (adds GPG key + apt source)
    print(_run('bash /tmp/add-of-repo.sh').stdout)

    # Step 4: Install OpenFOAM
    print('--- Installing openfoam2212-default (takes 5-8 min) ---')
    print(_run('apt-get install -y openfoam2212-default 2>&1 | tail -10').stdout)

    # Step 5: Verify
    r = _run('bash -c "source /usr/lib/openfoam/openfoam2212/etc/bashrc && foamVersion"',
             check=False)
    if r.returncode == 0:
        print(f'OpenFOAM version: {r.stdout.strip()}')
        print('Installation complete.')
    else:
        print('WARNING: foamVersion check failed — check output above.')
        print(r.stdout)

In [ ]:
# ── Step 3: Load OpenFOAM environment ─────────────────────────────────
import subprocess, os, sys, time

OF_BASHRC = '/usr/lib/openfoam/openfoam2212/etc/bashrc'

if not os.path.isfile(OF_BASHRC):
    raise RuntimeError(
        "OpenFOAM bashrc not found — install in cell 2 did not complete.\n"
        "→ Runtime → Disconnect and delete runtime → Runtime → Run all"
    )

_raw = subprocess.run(
    f'bash -c "source {OF_BASHRC} && env"',
    shell=True, capture_output=True, text=True
)
OF_ENV = {}
for _line in _raw.stdout.splitlines():
    _k, _, _v = _line.partition('=')
    if _k:
        OF_ENV[_k] = _v

_ver = OF_ENV.get('WM_PROJECT_VERSION', '')
_dir = OF_ENV.get('WM_PROJECT_DIR', '')
if not _ver:
    raise RuntimeError(
        "OpenFOAM environment did not load — WM_PROJECT_VERSION is unset.\n"
        "→ Runtime → Disconnect and delete runtime → Runtime → Run all"
    )
print(f'OpenFOAM {_ver} ready  ({_dir})')


def run_of(cmd, cwd=None):
    """Run a quick OpenFOAM command and return output (no streaming)."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True, env=OF_ENV
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'OpenFOAM command failed: {cmd}\n'
            f'STDOUT: {result.stdout[-2000:]}\n'
            f'STDERR: {result.stderr[-2000:]}'
        )
    return result.stdout


def run_of_stream(cmd, cwd=None):
    """
    Run a long OpenFOAM command streaming output line-by-line.
    If it fails, the last 60 lines are included in the error message
    so you don't need to scroll up to find the FOAM FATAL ERROR.
    """
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=OF_ENV, bufsize=1
    )
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    proc.wait()
    elapsed = time.time() - t0
    if proc.returncode != 0:
        tail = ''.join(lines[-60:])
        raise RuntimeError(
            f'\n{"="*60}\n'
            f'OpenFOAM command failed (rc={proc.returncode}): {cmd}\n'
            f'{"="*60}\n'
            f'--- Last 60 lines (contains the FOAM FATAL ERROR) ---\n'
            f'{tail}'
        )
    print(f'\n✓ {cmd.split()[0]} finished in {elapsed/60:.1f} min')
    return ''.join(lines)

In [ ]:
# ── Step 4: Generate propeller STL geometry ─────────────────────────────
!pip install -q numpy
!python geometry/generate_stl.py

import os
stl_files = os.listdir('geometry/stl')
print('Generated STL files:', stl_files)

In [ ]:
# ── Step 5: Create OpenFOAM case ───────────────────────────────────────
# ┌─────────────────────────────────────────────────────────────────────┐
# │  SIMULATION PARAMETERS — edit these to change what gets simulated   │
# └─────────────────────────────────────────────────────────────────────┘

CONFIG    = 'toroidal_medium_gap'   # ← propeller: conventional | toroidal_low_gap
                                    #              toroidal_medium_gap | toroidal_high_gap
CASE_TYPE = 'wind_tunnel_openjet'   # ← domain:    wind_tunnel_openjet (matches experiment)
                                    #              static_thrust (V_inf=0, open domain)
RPM       = 4000                    # ← RPM: 1000–6000
V_INF     = 9.0                     # ← freestream [m/s]: 0.0 / 3.0 / 9.0 / 15.0
N_PROCS   = 2                       # ← CPUs: 2 on Colab free, up to 8 on Colab Pro

# ┌─────────────────────────────────────────────────────────────────────┐
# │  MESH QUALITY — change this one variable to switch modes            │
# │                                                                     │
# │  'colab_fast'  →  ~30 min meshing, no boundary layers              │
# │                   Good for first run / workflow validation          │
# │                                                                     │
# │  'publication' →  ~3–4 hrs meshing, 5 boundary layers on propeller │
# │                   Use this for final thesis/paper results           │
# │                   Recommended: run on a workstation, not Colab      │
# └─────────────────────────────────────────────────────────────────────┘
MESH_QUALITY = 'colab_fast'   # ← CHANGE THIS to 'publication' for final results

# ── Generate base case from template ───────────────────────────────────
import os, subprocess
!python openfoam/generate_case.py \
    --config {CONFIG} \
    --case_type {CASE_TYPE} \
    --rpm {RPM} \
    --v_inf {V_INF} \
    --n_procs {N_PROCS}

CASE_DIR = f'runs/{CONFIG}_{int(RPM)}rpm_{CASE_TYPE}_{int(V_INF)}ms'

# ── Apply mesh quality settings to snappyHexMeshDict ───────────────────
shm_path = os.path.join(CASE_DIR, 'system', 'snappyHexMeshDict')
with open(shm_path) as f:
    shm = f.read()

if MESH_QUALITY == 'colab_fast':
    # Disable boundary layers (the main 2-hour cost on Colab)
    shm = shm.replace('addLayers       true',  'addLayers       false')

    # Reduce total cell count: 5M → 1.5M  (← increase for higher accuracy)
    shm = shm.replace('maxGlobalCells      5000000', 'maxGlobalCells      1500000')
    shm = shm.replace('maxLocalCells       1000000', 'maxLocalCells        500000')

    # Coarser surface refinement: (5 7) → (3 5)
    # ← Change back to (5 7) for publication quality
    shm = shm.replace('level (5 7)', 'level (3 5)')

    # Coarser distance-based refinement around propeller
    # ← Change back to ((0.005 6) (0.020 5) (0.050 4)) for publication quality
    shm = shm.replace(
        'levels ((0.005 6) (0.020 5) (0.050 4))',
        'levels ((0.010 4) (0.030 3))'
    )
    # Also patch static_thrust and wind_tunnel variants if present
    shm = shm.replace(
        'levels ((0.005 6) (0.020 5) (0.060 4))',
        'levels ((0.010 4) (0.030 3))'
    )

    print('Mesh quality : COLAB FAST')
    print('  addLayers  : false  (re-enable by setting MESH_QUALITY="publication")')
    print('  Max cells  : 1,500,000')
    print('  Refinement : level (3 5)')
    print('  snappyHexMesh expected time : ~15–30 min on Colab')

elif MESH_QUALITY == 'publication':
    # All settings left at template defaults (full quality)
    print('Mesh quality : PUBLICATION')
    print('  addLayers  : true  (5 boundary layers on propeller)')
    print('  Max cells  : 5,000,000')
    print('  Refinement : level (5 7)')
    print('  snappyHexMesh expected time : 3–4 hours (workstation recommended)')

else:
    raise ValueError(f'Unknown MESH_QUALITY: {MESH_QUALITY!r}. Use "colab_fast" or "publication".')

with open(shm_path, 'w') as f:
    f.write(shm)

print(f'\nCase directory : {CASE_DIR}')
!ls {CASE_DIR}

In [ ]:
# ── Step 6: Run blockMesh ──────────────────────────────────────────────
# Expected time: ~10-30 seconds
print('Running blockMesh...')
run_of_stream('blockMesh', cwd=CASE_DIR)

In [ ]:
# ── Step 7: Mesh — snappyHexMesh + rotatingZone + checkMesh ──────────
# snappyHexMesh: 15–30 min (colab_fast) or 3–4 hrs (publication)
# After meshing, topoSet unconditionally creates rotatingZone so MRF
# always works regardless of template settings.

print('Running snappyHexMesh...')
run_of_stream('snappyHexMesh -overwrite', cwd=CASE_DIR)

# ── Always create rotatingZone via topoSet ─────────────────────────────
# Do NOT rely on snappyHexMesh cellZone — topoSet is explicit and reliable.
# Cylinder dimensions must match MRFProperties (see constant/MRFProperties).
# If you change the MRF zone size, update point1/point2/radius here too.  ←
_toposet = """\
FoamFile { version 2.0; format ascii; class dictionary; object topoSetDict; }
actions
(
    { name rotatingZone; type cellZoneSet; action new;
      source cylinderToCell;
      point1 (0 0 -0.18);   // ← axial start of MRF zone [m]
      point2 (0 0  0.18);   // ← axial end   of MRF zone [m]
      radius  0.163; }      // ← radius of MRF zone [m] (1.6 × r_tip)
);
"""
with open(os.path.join(CASE_DIR, 'system', 'topoSetDict'), 'w') as f:
    f.write(_toposet)
run_of_stream('topoSet', cwd=CASE_DIR)
print('rotatingZone cellZone created')

# ── Check mesh quality ─────────────────────────────────────────────────
out = run_of('checkMesh', cwd=CASE_DIR)
for line in out.split('\n'):
    if any(k in line for k in ['cells:', 'faces:', 'points:', 'Max skewness',
                                'Max non-ortho', 'Overall', 'FOAM FATAL']):
        print(line.strip())

In [ ]:
# ── Step 8: Run simpleFoam ─────────────────────────────────────────────
# Expected time: 20–60 min (colab_fast mesh) or 2–4 hrs (publication mesh).
# Residuals stream below — watch for them to drop below 1e-4.
import shutil, glob, os

# Copy initial conditions (only if 0/ doesn't exist yet)
orig = os.path.join(CASE_DIR, '0.orig')
zero = os.path.join(CASE_DIR, '0')
if os.path.isdir(orig) and not os.path.isdir(zero):
    shutil.copytree(orig, zero)

if N_PROCS > 1:
    # Clean any stale processor dirs before decomposing
    # (stale dirs cause silent mesh mismatch errors)
    for _d in glob.glob(os.path.join(CASE_DIR, 'processor*')):
        shutil.rmtree(_d)

    run_of_stream('decomposePar', cwd=CASE_DIR)

    # Flags required for Colab:
    #   --allow-run-as-root : Colab always runs as root
    #   --oversubscribe     : OpenMPI may not detect all Colab CPU slots
    run_of_stream(
        f'mpirun --allow-run-as-root --oversubscribe -np {N_PROCS} '
        f'simpleFoam -parallel 2>&1',
        cwd=CASE_DIR
    )
    run_of_stream('reconstructPar', cwd=CASE_DIR)
else:
    run_of_stream('simpleFoam', cwd=CASE_DIR)

## Next Steps

### Run all cases
```bash
# Generate all 48 cases (4 configs × 3 RPMs × 4 wind speeds)
bash scripts/setup_all_cases.sh

# Run all cases (sequential on local machine)
bash scripts/run_all_cases.sh

# Post-process all cases
python postprocessing/extract_forces.py --runs_dir runs/

# Generate comparison plots (CFD + BEMT + Experimental)
python postprocessing/plot_comparison.py --source both --runs_dir runs/
```

### Replace parametric geometry with actual CAD
1. Export your propeller CAD as STL (units: metres)
2. Copy to `geometry/stl/<config_name>.stl`
3. Re-run the case setup and simulation

### Increase mesh resolution
Edit `openfoam/templates/*/system/snappyHexMeshDict`:
- Increase `level` values in `refinementSurfaces` (e.g., `(6 8)` for finer near-wall mesh)
- Increase background grid in `blockMeshDict` (e.g., `(30 30 40)` for denser background)

### Transient simulation (sliding mesh)
Replace `simpleFoam` with `pimpleFoam` and change MRF to AMI (Arbitrary Mesh Interface) for higher-fidelity time-resolved predictions.

In [ ]:
# ── Step 9: Extract CFD forces → CSV ──────────────────────────────────
# Reads OpenFOAM postProcessing/propellerForces output and writes
# results/cfd_results.csv with one row per completed case.

import sys, os
sys.path.insert(0, REPO_DIR)

from postprocessing.extract_forces import extract_all

rows = extract_all(runs_dir=os.path.join(REPO_DIR, 'runs'))

if rows:
    import pandas as pd
    df = pd.DataFrame(rows)
    print(df[['case', 'rpm', 'v_inf_ms', 'T_gf', 'Q_Nm', 'CP', 'eta']].to_string(index=False))
else:
    print('No completed cases found yet — run Step 8 first.')

In [ ]:
# ── Step 10: Plot — CFD vs BEMT vs Experimental ───────────────────────
# Generates:
#   1. Thrust vs RPM  (4 panels: static + 3 m/s + 9 m/s + 15 m/s)
#   2. CT vs J        (advance ratio sweep at 4000 RPM)
#   3. Efficiency vs J
#   4. CFD ★ overlaid on BEMT lines (if CFD results exist)
#
# All plots are saved to results/plots/ and also displayed inline below.

import matplotlib
matplotlib.use('Agg')   # works headless on Colab
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, sys

sys.path.insert(0, REPO_DIR)
from postprocessing.plot_comparison import main as run_plots

# Determine whether CFD results are available
cfd_csv = os.path.join(REPO_DIR, 'results', 'cfd_results.csv')
source  = 'both' if os.path.isfile(cfd_csv) else 'bemt'
print(f'Plotting source: {source}  ({"CFD + BEMT" if source == "both" else "BEMT only — run Steps 8-9 first for CFD overlay"})')

run_plots(source=source, runs_dir=os.path.join(REPO_DIR, 'runs'))

# Display plots inline
plots_dir = os.path.join(REPO_DIR, 'results', 'plots')
for fname in ['thrust_comparison.png', 'CT_vs_J.png', 'efficiency_vs_J.png', 'cfd_vs_bemt_vs_exp.png']:
    fpath = os.path.join(plots_dir, fname)
    if os.path.isfile(fpath):
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(mpimg.imread(fpath))
        ax.axis('off')
        ax.set_title(fname)
        plt.tight_layout()
        plt.show()
        print(f'↑ {fname}')